In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, dayofmonth

In [7]:
spark = SparkSession.builder \
    .appName("TVSalesAnalysis") \
    .master("local[*]") \
    .getOrCreate()

In [8]:
df_customers = spark.read.parquet("gold/user_profiles_enriched")
df_sales = spark.read.parquet("silver/sales")

In [9]:
df_customers.printSchema()
df_sales.printSchema()

root
 |-- client_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- phone_number: string (nullable = true)

root
 |-- client_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- purchase_date: date (nullable = true)



In [10]:
df_sales = df_sales.withColumn("purchase_date", to_date(col("purchase_date"), "yyyy-MM-d"))
df_customers = df_customers.withColumn("birth_date", to_date(col("birth_date"), "yyyy-MM-dd"))

In [12]:
from pyspark.sql.functions import floor, datediff, col
df_joined = df_sales.join(df_customers, on="client_id", how="inner")

df_joined = df_joined.withColumn(
    "age", 
    floor(datediff(col("purchase_date"), col("birth_date")) / 365.25)
)

In [14]:
df_tv = df_joined.filter(
    (col("product_name").rlike("(?i)television|tv")) &   # ігноруємо регістр
    (col("age") > 20) &
    (col("age") < 30) &
    (month(col("purchase_date")) == 9) &
    (dayofmonth(col("purchase_date")) <= 10)
)

In [15]:
df_result = df_tv.groupBy("state").count().orderBy(col("count").desc())

In [16]:
df_result.show(1)

[Stage 5:>                                                          (0 + 1) / 1]

+-----+-----+
|state|count|
+-----+-----+
|Idaho|  151|
+-----+-----+
only showing top 1 row
